# Round 2 - Notebook B: the final system under the reviewers' protocol

| Phase | Reviewer | What it produces |
|---|---|---|
| P2 | R2.4 | **blind BEA-2019 test** predictions (`bea19_test_submission.zip`, upload to CodaBench) + reproduction check on BEA dev |
| P3 | R1.4, R2.3 | length penalty x refinement x threshold, **on the full-budget checkpoints** |
| P4 | R1.4, R2.3 | no-edit pairs at full budget (stage 1 decoded like run 2; compared offline with run 2) |
| P5 | R1.5, R2.2 | **GEC-specific baselines** (gec-t5-small/base-clang8, GECToR-RoBERTa-base) on identical splits + official scorers |
| P6 | R1.6 | Qwen2.5-1.5B **fair re-run**: training-set exemplars by a stated rule, pinned revision, retokenized outputs |
| P7 | - | JFLEG with the **official** GLEU (the paper used NLTK's MT GLEU) |
| P8 | R2.5 | latency + memory for every system on the **same T4** |
| P9 | R2.6 | blind LLM-judge check of "harm" (reference-free) + an optional blinded sheet for a human check |

**Setup (Kaggle):** GPU **T4 x2** - Internet **on** - Add Input -> *Your Work* -> the **`t5v3_final_system`** version that produced the
paper's numbers (it contains `t5v3_model/`, `t5v3_model_stage2/`, `test.parquet`). Without it, P2-P4 and our own P7/P8 rows are skipped.

1. `QUICK_TEST = True`, *Save & Run All* (~30 min) -> check the last cell. 2. `QUICK_TEST = False`, *Save & Run All* (~5 h).
3. Download **`r2B_results.zip`** and **`bea19_test_submission.zip`**, send `r2B_results.zip` back.

In [ ]:
# ============================== CONFIG ==============================
QUICK_TEST = False
SESSION_BUDGET_H = 11.6
RESULTS_NAME, EXPORT_NAME = "r2B_results.json", "r2B_results.zip"
RUN = dict(bea_test=True, decode_factors=True, noedit_factor=True, gec_baselines=True,
           qwen=True, jfleg=True, latency=True, judge=True)
C4_EVAL_N, SUBSET_SEED = (5_000 if not QUICK_TEST else 300), 2026        # same subset as notebook A
LPS_C4, LPS_BENCH = (0.7, 1.0, 1.2), (0.7, 1.0)
TAUS = (None, 0.0, 0.05, 0.1, 0.2)                                       # None = no threshold; paper uses 0.0
GEC_BASELINES = ["gotutiyan/gec-t5-small-clang8", "gotutiyan/gec-t5-base-clang8", "gotutiyan/gector-roberta-base-5k"]
LLM_NAME, LLM_MAX_NEW, LLM_BATCH = "Qwen/Qwen2.5-1.5B-Instruct", 96, 32
JUDGE_NAME, JUDGE_FALLBACK = "Qwen/Qwen2.5-7B-Instruct", "Qwen/Qwen2.5-3B-Instruct"
JUDGE_N = {"harm": 150, "lateral": 150, "improved": 100} if not QUICK_TEST else {"harm": 8, "lateral": 8, "improved": 4}
LAT_N, LAT_WARM = (300, 10) if not QUICK_TEST else (30, 3)


In [ ]:
# ---------------------------------------------------------------- installs
import subprocess, sys, importlib
def pip(*a):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a], capture_output=True, text=True)
    if r.returncode: print("pip warning:", " ".join(a), r.stderr[-300:])
pip("errant==3.0.2", "sentencepiece", "pyarrow")
import spacy
try:
    spacy.load("en_core_web_sm")
except OSError:
    v = ".".join(spacy.__version__.split(".")[:2]) + ".0"
    pip(f"https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-{v}/en_core_web_sm-{v}-py3-none-any.whl")
spacy.load("en_core_web_sm")
print("spaCy", spacy.__version__, "| en_core_web_sm OK")


In [ ]:
# ---------------------------------------------------------------- imports + run bookkeeping
import os, re, io, csv, json, gc, math, time, random, shutil, tarfile, zipfile, urllib.request, contextlib
from pathlib import Path
from collections import Counter
from difflib import SequenceMatcher
import numpy as np, pandas as pd, pyarrow as pa, pyarrow.parquet as pq
import torch
from torch.utils.data import Dataset
from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM,
                          DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments)
os.environ["WANDB_DISABLED"] = "true"
OUT = Path("/kaggle/working"); EV = OUT / "eval_data"; EV.mkdir(exist_ok=True)
T0 = time.time()
def hours_left(): return SESSION_BUDGET_H - (time.time() - T0) / 3600
def budget_ok(need_h, what=""):
    ok = hours_left() > need_h
    if not ok: print(f"!! budget guard: {hours_left():.2f} h left < {need_h} h needed -> skipping {what}")
    return ok
RES_PATH = OUT / RESULTS_NAME
RESULTS = json.loads(RES_PATH.read_text()) if RES_PATH.exists() else {}
def save_results(): RES_PATH.write_text(json.dumps(RESULTS, indent=1, default=str))
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device={DEV} | GPUs={torch.cuda.device_count()} | budget={SESSION_BUDGET_H} h | QUICK_TEST={QUICK_TEST}")


In [ ]:
# ---------------------------------------------------------------- official evaluation data + scorers
CONLL_URL = "https://www.comp.nus.edu.sg/~nlp/conll14st/conll14st-test-data.tar.gz"
M2_URL    = "https://www.comp.nus.edu.sg/~nlp/sw/m2scorer.tar.gz"
BEA_URL   = "https://www.cl.cam.ac.uk/research/nl/bea2019st/data/wi+locness_v2.1.bea19.tar.gz"
JF_BASE   = "https://raw.githubusercontent.com/keisks/jfleg/master/"

def fetch(url, dest):
    if not Path(dest).exists(): urllib.request.urlretrieve(url, dest)
    return Path(dest)

def untar(tgz, into):
    with tarfile.open(tgz) as t: t.extractall(into)

# CoNLL-2014 test (official, 1,312 sentences, 2 annotators)
if not (EV / "conll14st-test-data").exists(): untar(fetch(CONLL_URL, EV / "conll.tgz"), EV)
CONLL_M2 = EV / "conll14st-test-data" / "noalt" / "official-2014.combined.m2"
# official MaxMatch scorer, ported to Python 3 OFFLINE and embedded: Kaggle now runs Python 3.13, which removed
# lib2to3, so converting it at run time is no longer possible. (Port verified: reproduces 0.4852 / 0.3096 exactly.)
import base64 as _b64, gzip as _gz
with tarfile.open(fileobj=io.BytesIO(_gz.decompress(_b64.b64decode("...")))) as _t:
    _t.extractall(EV)                                   # always overwrite with the ported copy
M2_SCRIPT = EV / "m2scorer" / "scripts" / "m2scorer.py"
# W&I+LOCNESS v2.1 (BEA-2019): dev gold M2, blind test input, train M2
if not (EV / "wi+locness").exists(): untar(fetch(BEA_URL, EV / "wi.tgz"), EV)
BEA_DEV_M2 = EV / "wi+locness" / "m2" / "ABCN.dev.gold.bea19.m2"
BEA_TEST_ORIG = EV / "wi+locness" / "test" / "ABCN.test.bea19.orig"
# JFLEG test (747 sentences, 4 references) + the OFFICIAL GLEU script (Napoles et al., 2016 version)
JF = EV / "jfleg"; JF.mkdir(exist_ok=True)
for n in ["test/test.src"] + [f"test/test.ref{k}" for k in range(4)] + ["eval/gleu.py"]:
    fetch(JF_BASE + n, JF / Path(n).name)
sys.path.insert(0, str(JF))
from gleu import GLEU

def m2_sources(path): return [l[2:].strip() for l in open(path, encoding="utf-8") if l.startswith("S ")]
def read_m2(path):
    out = []
    for block in Path(path).read_text(encoding="utf-8").strip().split("\n\n"):
        lines = block.split("\n"); src = lines[0][2:].strip(); ed = {}
        for ln in lines[1:]:
            if not ln.startswith("A "): continue
            f = ln[2:].split("|||"); s, e = map(int, f[0].split()); ann = int(f[-1])
            ed.setdefault(ann, [])
            if f[1] != "noop": ed[ann].append((s, e, f[2]))
        out.append((src, ed))
    return out
def apply_m2_edits(src, edits):
    toks = src.split(); off = 0
    for s, e, cor in sorted(edits, key=lambda x: (x[0], x[1])):
        c = cor.split() if cor and cor != "-NONE-" else []
        toks[s + off:e + off] = c; off += len(c) - (e - s)
    return " ".join(toks)

CONLL_SRC = m2_sources(CONLL_M2)
if QUICK_TEST:                       # smoke test: first 200 CoNLL sentences with the matching gold
    _b = Path(CONLL_M2).read_text(encoding="utf-8").strip().split("\n\n")[:200]
    CONLL_M2 = EV / "conll_quick.m2"; CONLL_M2.write_text("\n\n".join(_b) + "\n\n", encoding="utf-8")
    CONLL_SRC = m2_sources(CONLL_M2)
BEA_DEV = read_m2(BEA_DEV_M2)
BEA_SRC = [s for s, _ in BEA_DEV]
BEA_TGT = [apply_m2_edits(s, ed.get(0, [])) for s, ed in BEA_DEV]      # first annotator (as in the paper)
BEA_TEST_SRC = [l.rstrip("\n") for l in open(BEA_TEST_ORIG, encoding="utf-8")]
JF_SRC = [l.rstrip("\n") for l in open(JF / "test.src", encoding="utf-8")]
JF_REFS = [JF / f"test.ref{k}" for k in range(4)]
if QUICK_TEST:
    BEA_N, JF_N = 200, 120
else:
    BEA_N, JF_N = len(BEA_SRC), len(JF_SRC)
print(f"CoNLL-2014 {len(CONLL_SRC)} | BEA dev {len(BEA_SRC)} | BEA test {len(BEA_TEST_SRC)} | JFLEG {len(JF_SRC)}")


In [ ]:
# ---------------------------------------------------------------- retokenizers + official scorers
# Both benchmarks ship PRE-TOKENIZED sources and both scorers compare whitespace tokens, so every
# system output is retokenized with the tokenizer that best reproduces the benchmark's own convention
# (round-trip test on the sources: regex 99.3% on CoNLL-2014, spaCy 98.7% on BEA-2019).
import spacy
_NLP = spacy.load("en_core_web_sm", disable=["parser", "ner", "lemmatizer", "tagger", "attribute_ruler"])
def spacy_tok(s): return " ".join(t.text for t in _NLP.make_doc(str(s)) if not t.is_space)
_CLITIC = re.compile(r"(?i)(\w)(n't|'s|'re|'ve|'ll|'d|'m)\b")
_PA = re.compile(r"(\S)([.,!?;:%)\]\"](?:\s|$))"); _PB = re.compile(r"(^|\s)([(\[\"])(\S)")
def regex_tok(s):
    s = _CLITIC.sub(r"\1 \2", str(s))
    for _ in range(3): s = _PA.sub(r"\1 \2", s)
    return re.sub(r"\s+", " ", _PB.sub(r"\1\2 \3", s)).strip()
RETOK = {"conll": regex_tok, "jfleg": regex_tok, "bea": spacy_tok}
def norm_ws(s): return re.sub(r"\s+", " ", str(s)).strip()

def official_m2(hyps, tag, gold=None):
    gold = gold or CONLL_M2
    n = sum(1 for l in open(gold, encoding="utf-8") if l.startswith("S "))
    assert len(hyps) == n, f"{tag}: {len(hyps)} hyps vs {n} gold"
    hp = OUT / f"_m2_{tag}.txt"; hp.write_text("\n".join(hyps) + "\n", encoding="utf-8")
    r = subprocess.run([sys.executable, str(M2_SCRIPT), str(hp), str(gold)], capture_output=True, text=True,
                       timeout=7200, env={**os.environ, "PYTHONUTF8": "1"})
    g = dict(re.findall(r"(Precision|Recall|F_0\.5)\s*:\s*([0-9.]+)", r.stdout))
    if not g: print("m2scorer failed:", r.stderr[-400:]); return {"error": r.stderr[-400:]}
    return {"P": float(g["Precision"]), "R": float(g["Recall"]), "F05": float(g["F_0.5"])}

import errant
from errant.commands import compare_m2
_ANN = {"a": None}
def ann():
    if _ANN["a"] is None: _ANN["a"] = errant.load("en")
    return _ANN["a"]
def errant_official(srcs, hyps, gold_m2, tag):
    """Hypothesis M2 via ERRANT (whitespace tokens, as errant_parallel), then the official errant_compare."""
    a = ann(); hp = OUT / f"_errant_{tag}.m2"
    with open(hp, "w", encoding="utf-8") as f:
        for s, h in zip(srcs, hyps):
            o = a.parse(s); ed = a.annotate(o, a.parse(h))
            f.write("S " + s + "\n")
            if not ed: f.write("A -1 -1|||noop|||-NONE-|||REQUIRED|||-NONE-|||0\n")
            for e in ed: f.write(e.to_m2() + "\n")
            f.write("\n")
    gold_use = gold_m2
    if len(srcs) < len(m2_sources(gold_m2)):                       # QUICK_TEST subsets
        blocks = Path(gold_m2).read_text(encoding="utf-8").strip().split("\n\n")[:len(srcs)]
        gold_use = OUT / f"_gold_{tag}.m2"; gold_use.write_text("\n\n".join(blocks) + "\n\n", encoding="utf-8")
    buf = io.StringIO(); old = sys.argv
    sys.argv = ["compare_m2", "-hyp", str(hp), "-ref", str(gold_use)]
    try:
        with contextlib.redirect_stdout(buf): compare_m2.main()
    finally: sys.argv = old
    found = re.findall(r"^(\d+)\s+(\d+)\s+(\d+)\s+([0-9.]+)\s+([0-9.]+)\s+([0-9.]+)\s*$", buf.getvalue(), re.M)
    if not found: print("errant_compare output not parsed"); return {"error": buf.getvalue()[-400:]}
    tp, fp, fn, p, r, f = found[-1]
    return {"TP": int(tp), "FP": int(fp), "FN": int(fn), "P": float(p), "R": float(r), "F05": float(f)}

def jfleg_gleu(hyps, n=None):
    """OFFICIAL JFLEG GLEU (keisks/jfleg eval/gleu.py, 500 random-reference iterations), source-aware."""
    n = n or len(hyps)
    sp, hp = OUT / "_jf_src.txt", OUT / "_jf_hyp.txt"
    sp.write_text("\n".join(JF_SRC[:n]) + "\n", encoding="utf-8"); hp.write_text("\n".join(hyps[:n]) + "\n", encoding="utf-8")
    refs = []
    for k, rp in enumerate(JF_REFS):
        p = OUT / f"_jf_ref{k}.txt"; p.write_text("\n".join(open(rp, encoding="utf-8").read().split("\n")[:n]) + "\n", encoding="utf-8"); refs.append(str(p))
    g = GLEU(4); g.load_sources(str(sp)); g.load_references(refs)
    mean, std, ci = list(g.run_iterations(num_iterations=500, source=str(sp), hypothesis=str(hp), per_sent=False))[0]
    return {"GLEU": float(mean), "std": float(std), "ci95": ci}

def lev(a, b):
    a, b = str(a).split(), str(b).split(); prev = list(range(len(b) + 1))
    for i, x in enumerate(a, 1):
        cur = [i]
        for j, y in enumerate(b, 1): cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (x != y)))
        prev = cur
    return prev[-1]
def outcome_table(srcs, tgts, preds):
    c = Counter(); n = len(srcs)
    for s, t, p in zip(srcs, tgts, preds):
        if p.strip() == t.strip(): c["perfect"] += 1
        elif p.strip() == s.strip(): c["unchanged"] += 1
        else:
            ds, dp = lev(s, t), lev(p, t)
            c["improved" if dp < ds else ("lateral" if dp == ds else "harm")] += 1
    o = {k: round(100 * c[k] / n, 2) for k in ("perfect", "improved", "unchanged", "lateral", "harm")}
    o["edited_pct"] = round(100 * sum(a.strip() != b.strip() for a, b in zip(srcs, preds)) / n, 2)
    return o
def single_ref_errant(srcs, tgts, preds):
    """Edit-level P/R/F0.5 against ONE reference (the in-domain C4 test), plus per-sentence counts."""
    a = ann(); tp = fp = fn = 0; per = []
    for s, t, p in zip(srcs, tgts, preds):
        o = a.parse(s)
        gold = set() if t == s else {(e.o_start, e.o_end, e.c_str) for e in a.annotate(o, a.parse(t)) if e.type != "noop"}
        hyp = set() if p == s else {(e.o_start, e.o_end, e.c_str) for e in a.annotate(o, a.parse(p)) if e.type != "noop"}
        x = (len(gold & hyp), len(hyp - gold), len(gold - hyp)); per.append(x)
        tp += x[0]; fp += x[1]; fn += x[2]
    P = tp / max(tp + fp, 1); R = tp / max(tp + fn, 1)
    return {"P": P, "R": R, "F05": (1.25 * P * R / (0.25 * P + R)) if (P + R) else 0.0, "TP": tp, "FP": fp, "FN": fn}, per
print("scorers ready")


In [ ]:
# ---------------------------------------------------------------- FLAN-T5 decoding, exactly as in the paper
PREFIX = "Correct the grammar in this sentence: "
MAX_INPUT_LEN = MAX_TARGET_LEN = 128
GEN_KWARGS = dict(max_length=128, num_beams=5, length_penalty=0.7, no_repeat_ngram_size=3,
                  early_stopping=True, min_length=1)
REFINE_MAX_EDITS = 3
GEN_BATCH = 64
def token_edit(a, b):
    sm = SequenceMatcher(None, str(a).split(), str(b).split())
    return sum(max(i2 - i1, j2 - j1) for tag, i1, i2, j1, j2 in sm.get_opcodes() if tag != "equal")
def load_t5(path):
    tok = AutoTokenizer.from_pretrained(str(path)); m = AutoModelForSeq2SeqLM.from_pretrained(str(path)).to(DEV).eval()
    return tok, m
@torch.no_grad()
def t5_gen(tok, m, texts, gen=None, bs=GEN_BATCH, desc=""):
    kw = dict(GEN_KWARGS); kw.update(gen or {}); out = []
    for i in range(0, len(texts), bs):
        enc = tok([PREFIX + str(s) for s in texts[i:i + bs]], return_tensors="pt", padding=True,
                  truncation=True, max_length=MAX_INPUT_LEN).to(m.device)
        out += tok.batch_decode(m.generate(**enc, **kw), skip_special_tokens=True)
        if desc and (i // bs) % 25 == 0: print(f"   {desc} {i}/{len(texts)}", flush=True)
    return out
def refine(tok, m, srcs, pass1, gen=None):
    need = [i for i, p in enumerate(pass1) if p.strip() != str(srcs[i]).strip()]
    out = list(pass1)
    if need:
        r = t5_gen(tok, m, [pass1[k] for k in need], gen)
        for k, i in enumerate(need):
            if token_edit(pass1[i], r[k]) <= REFINE_MAX_EDITS: out[i] = r[k]
    return out
@torch.no_grad()
def mean_nll(tok, m, ins, outs, bs=GEN_BATCH):
    res = []
    for i in range(0, len(ins), bs):
        enc = tok([PREFIX + str(s) for s in ins[i:i + bs]], return_tensors="pt", padding=True, truncation=True,
                  max_length=MAX_INPUT_LEN).to(m.device)
        lab = tok(text_target=[str(t) for t in outs[i:i + bs]], return_tensors="pt", padding=True, truncation=True,
                  max_length=MAX_TARGET_LEN).to(m.device)
        labels = lab["input_ids"].masked_fill(lab["attention_mask"] == 0, -100)
        lp = torch.log_softmax(m(**enc, labels=labels).logits.float(), dim=-1)
        nll = -lp.gather(2, labels.clamp(min=0).unsqueeze(-1)).squeeze(-1); mask = (labels != -100).float()
        res += ((nll * mask).sum(1) / mask.sum(1).clamp(min=1)).tolist()
    return res
def margins(tok, m, srcs, preds):
    idx = [i for i, (s, p) in enumerate(zip(srcs, preds)) if str(p).strip() != str(s).strip()]
    mg = [None] * len(srcs)
    if idx:
        a = mean_nll(tok, m, [srcs[i] for i in idx], [preds[i] for i in idx])
        b = mean_nll(tok, m, [srcs[i] for i in idx], [srcs[i] for i in idx])
        for k, i in enumerate(idx): mg[i] = b[k] - a[k]
    return mg
def revert(srcs, preds, mg, tau):
    if tau is None: return list(preds)
    return [s if (x is not None and x < tau) else p for s, p, x in zip(srcs, preds, mg)]
def decode_variants(tok, m, srcs, lps=(0.7,), refines=(True,), taus=(0.0,), desc=""):
    """All (length-penalty x refinement x threshold) variants from ONE beam decode per length penalty."""
    out = {}
    for lp in lps:
        p1 = t5_gen(tok, m, srcs, {"length_penalty": lp}, desc=f"{desc} lp={lp}")
        for rf in refines:
            p = refine(tok, m, srcs, p1, {"length_penalty": lp}) if rf else list(p1)
            mg = margins(tok, m, srcs, p) if any(t is not None for t in taus) else None
            for tau in taus:
                out[(lp, rf, tau)] = [norm_ws(x) for x in revert(srcs, p, mg, tau)]
    return out
def vkey(lp, rf, tau): return f"lp{lp}_ref{int(rf)}_tau{'none' if tau is None else tau}"


In [ ]:
STORED = json.loads(__import__("gzip").decompress(__import__("base64").b64decode("...")))
print({k: len(v) for k, v in STORED.items()})


In [ ]:
# ---------------------------------------------------------------- the paper's final checkpoints
# Add Input -> "Your Work" -> the t5v3_final_system notebook output. Every model folder found is listed; if the final
# folders t5v3_model / t5v3_model_stage2 are absent, the trainer's BEST checkpoint (lowest validation loss) is used.
MODELS = []
for root, dirs, files in os.walk("/kaggle/input", followlinks=True):
    if "config.json" in files and any(f in files for f in ("model.safetensors", "pytorch_model.bin")):
        MODELS.append(Path(root))
print("model folders under /kaggle/input:")
for mdl in MODELS: print("   ", mdl)
for top in sorted(Path("/kaggle/input").glob("*/*/*")) if Path("/kaggle/input").exists() else []:
    if top.is_dir(): print("  input dir:", top, "->", sorted(p.name for p in top.iterdir())[:40])
def pick(final_name, ckpt_parent):
    for mdl in MODELS:
        if mdl.name == final_name: return mdl, "final folder"
    cks = [mdl for mdl in MODELS if mdl.parent.name == ckpt_parent and mdl.name.startswith("checkpoint-")]
    if not cks: return None, "missing"
    newest = max(cks, key=lambda p: int(p.name.split("-")[-1]))
    st = newest / "trainer_state.json"
    best = json.loads(st.read_text()).get("best_model_checkpoint") if st.exists() else None
    for mdl in cks:
        if best and mdl.name == Path(best).name: return mdl, "best checkpoint (trainer_state)"
    return newest, "newest checkpoint (no trainer_state)"
(S1_DIR, S1_HOW), (S2_DIR, S2_HOW) = pick("t5v3_model", "t5v3_ckpts"), pick("t5v3_model_stage2", "t5v3_ckpts_stage2")
if S1_DIR is not None and (S1_DIR.parent / "test.parquet").exists():
    TEST_PQ = S1_DIR.parent / "test.parquet"
else:
    TEST_PQ = next(iter(Path("/kaggle/input").rglob("test.parquet")), None) if Path("/kaggle/input").exists() else None
HAVE_CKPT = S1_DIR is not None and S2_DIR is not None
RESULTS["checkpoints"] = {"stage1": str(S1_DIR), "stage1_how": S1_HOW, "stage2": str(S2_DIR), "stage2_how": S2_HOW,
                          "all_model_folders": [str(x) for x in MODELS]}; save_results()
print("stage-1:", S1_DIR, f"[{S1_HOW}]", "| stage-2:", S2_DIR, f"[{S2_HOW}]", "| test.parquet:", TEST_PQ)
if not HAVE_CKPT:
    print("!! Final checkpoints NOT found -- see the folder list above and send it to Claude. Baselines, Qwen,")
    print("!! JFLEG do-nothing and the judge (on the paper's stored predictions) still run.")
C4_SRC = C4_TGT = SUB_IDX = None
if TEST_PQ is not None:
    te = pq.read_table(TEST_PQ).to_pandas()
    SUB_IDX = np.sort(np.random.default_rng(SUBSET_SEED).permutation(len(te))[:C4_EVAL_N])
    C4_SRC = [te.src[i] for i in SUB_IDX]; C4_TGT = [te.tgt[i] for i in SUB_IDX]; del te


In [ ]:
# ---------------------------------------------------------------- P2: blind BEA-2019 test + reproduction check
# Final configuration of the paper: stage-2 model, beam 5, lp 0.7, refinement, confidence threshold tau = 0.0
if RUN["bea_test"] and HAVE_CKPT and "bea_test" not in RESULTS and budget_ok(0.7, "BEA test"):
    tok, m = load_t5(S2_DIR)
    def final(srcs, name): return decode_variants(tok, m, srcs, (0.7,), (True,), (0.0,), desc=name)[(0.7, True, 0.0)]
    bs = BEA_SRC[:BEA_N]; p = final(bs, "bea-dev"); pr = [RETOK["bea"](x) for x in p]
    RESULTS["bea_dev_reproduction"] = {"errant": errant_official(bs, pr, BEA_DEV_M2, "bea_repro"),
        "paper_reported_F05": 0.4288,
        "identical_to_stored_outputs_pct": round(100 * np.mean([a == b for a, b in zip(p, STORED["bea_dev_pred"][:len(p)])]), 2)}
    pd.DataFrame({"src": bs, "pred": p, "pred_retok": pr}).to_csv(OUT / "pred_bea_dev_final.csv", index=False)
    tsrc = BEA_TEST_SRC if not QUICK_TEST else BEA_TEST_SRC[:50]
    p = final(tsrc, "bea-test"); pr = [RETOK["bea"](x) for x in p]
    (OUT / "bea19_test_hyp.txt").write_text("\n".join(pr) + "\n", encoding="utf-8")
    with zipfile.ZipFile(OUT / "bea19_test_submission.zip", "w") as z: z.write(OUT / "bea19_test_hyp.txt", "bea19_test_hyp.txt")
    RESULTS["bea_test"] = {"n": len(pr), "file": "bea19_test_submission.zip", "edited_pct": outcome_table(tsrc, tsrc, pr)["edited_pct"]}
    save_results(); print(RESULTS["bea_dev_reproduction"]); del m; gc.collect(); torch.cuda.empty_cache()


In [ ]:
# ---------------------------------------------------------------- P3: decode-only factors at FULL budget
# length penalty x refinement x confidence threshold, on the paper's final checkpoints (R1.4 / R2.3)
if RUN["decode_factors"] and HAVE_CKPT and "factors_c4" not in RESULTS and C4_SRC is not None and budget_ok(1.3, "decode factors"):
    tok, m = load_t5(S1_DIR)
    V = decode_variants(tok, m, C4_SRC, LPS_C4, (False, True), TAUS, desc="stage1-c4")
    rows, res = {}, {}
    for (lp, rf, tau), p in V.items():
        k = vkey(lp, rf, tau); mm, _ = single_ref_errant(C4_SRC, C4_TGT, p)
        res[k] = {**mm, **outcome_table(C4_SRC, C4_TGT, p)}; rows[k] = p
    pd.DataFrame({"test_index": SUB_IDX, "src": C4_SRC, "tgt": C4_TGT, **rows}).to_csv(OUT / "pred_c4_stage1_variants.csv", index=False)
    RESULTS["factors_c4"] = res; save_results(); del m; gc.collect(); torch.cuda.empty_cache()
if RUN["decode_factors"] and HAVE_CKPT and "factors_bench" not in RESULTS and budget_ok(1.0, "benchmark factors"):
    tok, m = load_t5(S2_DIR); res = {}
    bs = BEA_SRC[:BEA_N]
    for name, srcs in (("bea_dev", bs), ("conll", CONLL_SRC)):
        V = decode_variants(tok, m, srcs, LPS_BENCH, (False, True), (None, 0.0, 0.1), desc=f"stage2-{name}")
        rows = {}
        for (lp, rf, tau), p in V.items():
            k = vkey(lp, rf, tau); pr = [RETOK["bea" if name == "bea_dev" else "conll"](x) for x in p]; rows[k] = pr
            res[f"{name}|{k}"] = (errant_official(bs, pr, BEA_DEV_M2, f"f_{name}_{k}") if name == "bea_dev"
                                  else official_m2(pr, f"f_{name}_{k}"))
        pd.DataFrame({"src": srcs, **rows}).to_csv(OUT / f"pred_{name}_stage2_variants.csv", index=False)
    RESULTS["factors_bench"] = res; save_results(); del m; gc.collect(); torch.cuda.empty_cache()


In [ ]:
# ---------------------------------------------------------------- P4: no-edit pairs at FULL budget
# stage-1 (trained WITH no-edit pairs) decoded exactly like run 2 (lp 0.7 + refinement, no threshold);
# compared offline, sentence by sentence, with run 2 (same data, same recipe, WITHOUT no-edit pairs).
if RUN["noedit_factor"] and HAVE_CKPT and "noedit_bench" not in RESULTS and budget_ok(0.6, "no-edit factor"):
    tok, m = load_t5(S1_DIR); res = {}
    for name, srcs in (("bea_dev", BEA_SRC[:BEA_N]), ("conll", CONLL_SRC)):
        p = [norm_ws(x) for x in refine(tok, m, srcs, t5_gen(tok, m, srcs, desc=f"stage1-{name}"))]
        pr = [RETOK["bea" if name == "bea_dev" else "conll"](x) for x in p]
        res[name] = (errant_official(srcs, pr, BEA_DEV_M2, f"ne_{name}") if name == "bea_dev" else official_m2(pr, f"ne_{name}"))
        pd.DataFrame({"src": srcs, "pred": p, "pred_retok": pr}).to_csv(OUT / f"pred_{name}_stage1_run2decode.csv", index=False)
    RESULTS["noedit_bench"] = res; save_results(); del m; gc.collect(); torch.cuda.empty_cache()


In [ ]:
# ---------------------------------------------------------------- P5: GEC-specific public baselines, same splits + scorers
pip("--no-deps", "git+https://github.com/gotutiyan/gec-t5", "git+https://github.com/gotutiyan/gector")
VERB = fetch("https://github.com/grammarly/gector/raw/master/data/verb-form-vocab.txt", EV / "verb-form-vocab.txt")
def load_gector(mid, device=DEV):
    """GECToR.__init__ calls AutoModel.from_pretrained itself, which recent transformers forbids inside
    GECToR.from_pretrained (meta-device init). Build from the config instead, then load the released weights."""
    from gector import GECToR, GECToRConfig
    from huggingface_hub import hf_hub_download
    m = GECToR(GECToRConfig.from_pretrained(mid))
    try:
        from safetensors.torch import load_file
        sd = load_file(hf_hub_download(mid, "model.safetensors"))
    except Exception:
        sd = torch.load(hf_hub_download(mid, "pytorch_model.bin"), map_location="cpu")
    missing, unexpected = m.load_state_dict(sd, strict=False)
    bad = [k for k in missing if "position_ids" not in k]
    assert not bad, f"GECToR weights missing: {bad[:5]}"
    print(f"  {mid}: loaded {len(sd)} tensors (missing {len(missing)}, unexpected {len(unexpected)})")
    return m.to(device).eval()
def run_baseline(mid, srcs):
    if "gector" in mid:
        from gector import predict, load_verb_dict
        tok = AutoTokenizer.from_pretrained(mid); m = load_gector(mid)
        enc, dec = load_verb_dict(str(VERB))
        out = predict(m, tok, srcs, enc, dec, keep_confidence=0.3, min_error_prob=0.6, n_iteration=5, batch_size=64)
        how = "gector.predict(keep_confidence=0.3, min_error_prob=0.6, n_iteration=5)"
    else:
        tok = AutoTokenizer.from_pretrained(mid); m = AutoModelForSeq2SeqLM.from_pretrained(mid).to(DEV).eval()
        try:
            from gec_t5 import generate as g5
            out = g5(model=m, tokenizer=tok, sources=srcs, batch_size=32); how = "gec_t5.generate (authors' defaults)"
        except Exception as e:
            out = []; how = f"model.generate(num_beams=5, max_length=128) [gec_t5 import failed: {e!r}]"
            with torch.no_grad():
                for i in range(0, len(srcs), 32):
                    x = tok(srcs[i:i + 32], return_tensors="pt", padding=True, truncation=True, max_length=128).to(DEV)
                    out += tok.batch_decode(m.generate(**x, num_beams=5, max_length=128), skip_special_tokens=True)
    del m; gc.collect(); torch.cuda.empty_cache()
    return [norm_ws(x) for x in out], how
if RUN["gec_baselines"]:
    for mid in GEC_BASELINES:
        key = f"baseline|{mid}"
        if key in RESULTS or not budget_ok(0.5, mid): continue
        r = {}; short = mid.split("/")[-1]
        try:
            for name, srcs in (("conll", CONLL_SRC), ("bea_dev", BEA_SRC[:BEA_N]), ("jfleg", JF_SRC[:JF_N])):
                p, how = run_baseline(mid, srcs); pr = [RETOK["bea" if name == "bea_dev" else name](x) for x in p]
                r[name] = (official_m2(pr, f"b_{short}") if name == "conll" else
                           errant_official(srcs, pr, BEA_DEV_M2, f"b_{short}") if name == "bea_dev" else jfleg_gleu(pr, JF_N))
                r[name] = r[name] or {"error": "scorer returned nothing"}
                r[name]["edited_pct"] = outcome_table(srcs, srcs, pr)["edited_pct"]; r["inference"] = how
                pd.DataFrame({"src": srcs, "pred": p, "pred_retok": pr}).to_csv(OUT / f"pred_{name}_{short}.csv", index=False)
        except Exception as e:
            r["error"] = repr(e)
        RESULTS[key] = r; save_results(); print(mid, json.dumps(r, default=str)[:600])


In [ ]:
# ---------------------------------------------------------------- P6: Qwen2.5-1.5B-Instruct, fair re-run (R1.6)
# Fixes from round 1: (i) exemplars drawn by a stated rule from W&I+LOCNESS TRAIN (round 1 used two BEA-dev sentences),
# (ii) model revision pinned and recorded, (iii) outputs retokenized like every other system.
from huggingface_hub import model_info
SYSTEM = ("You are a grammatical error correction system. Return ONLY the corrected sentence. "
          "Make the minimum number of edits. If the sentence is already correct, repeat it unchanged.")
def pick_exemplars():
    pool = []
    for p in sorted((EV / "wi+locness" / "m2").glob("*.train.gold.bea19.m2")):
        for s, ed in read_m2(p):
            e0 = ed.get(0, []); t = apply_m2_edits(s, e0); n = len(s.split())
            if 8 <= n <= 20 and 1 <= len(e0) <= 3 and s != t and s.isascii(): pool.append((s, t))
    rng = random.Random(2024); ex = rng.sample(pool, 4)
    banned = {re.sub(r"\W+", " ", x.lower()).strip() for x in BEA_SRC + CONLL_SRC}
    assert not any(re.sub(r"\W+", " ", s.lower()).strip() in banned for s, _ in ex), "exemplar overlaps an evaluation set"
    return ex, len(pool)
def prompts(tok, srcs, shots):
    out = []
    for s in srcs:
        msgs = [{"role": "system", "content": SYSTEM}]
        for a, b in shots: msgs += [{"role": "user", "content": a}, {"role": "assistant", "content": b}]
        msgs.append({"role": "user", "content": s})
        out.append(tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True))
    return out
@torch.no_grad()
def llm_gen(m, tok, srcs, shots):
    P = prompts(tok, srcs, shots); out = []
    for i in range(0, len(P), LLM_BATCH):
        enc = tok(P[i:i + LLM_BATCH], return_tensors="pt", padding=True).to(m.device)
        g = m.generate(**enc, max_new_tokens=LLM_MAX_NEW, do_sample=False, pad_token_id=tok.pad_token_id)
        out += [norm_ws(tok.decode(x[enc.input_ids.shape[1]:], skip_special_tokens=True).split("\n")[0]) for x in g]
    return out
if RUN["qwen"] and "qwen" not in RESULTS and budget_ok(1.0, "Qwen"):
    EX, pool_n = pick_exemplars(); rev = model_info(LLM_NAME).sha
    tok = AutoTokenizer.from_pretrained(LLM_NAME, revision=rev); tok.padding_side = "left"
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    m = AutoModelForCausalLM.from_pretrained(LLM_NAME, revision=rev, torch_dtype=torch.float16, device_map={"": 0}).eval()
    r = {"model": LLM_NAME, "revision": rev, "dtype": "float16 (no quantization)", "decoding": "greedy, max_new_tokens=96",
         "postprocessing": "first output line, whitespace-normalized, then benchmark retokenizer", "system_prompt": SYSTEM,
         "exemplar_rule": "W&I+LOCNESS train (first annotator), 8-20 tokens, 1-3 edits, ASCII; random.Random(2024).sample(pool, 4)",
         "exemplar_pool_size": pool_n, "exemplars": EX}
    for shots_name, shots in (("zeroshot", []), ("fourshot", EX)):
        for name, srcs in (("conll", CONLL_SRC), ("bea_dev", BEA_SRC[:BEA_N])):
            p = llm_gen(m, tok, srcs, shots); pr = [RETOK["bea" if name == "bea_dev" else "conll"](x) for x in p]
            s = (official_m2(pr, f"q_{shots_name}") if name == "conll" else errant_official(srcs, pr, BEA_DEV_M2, f"q_{shots_name}"))
            s = s or {"error": "scorer returned nothing"}; s.update(outcome_table(srcs, srcs, pr)); s["raw_glued_punct_pct"] = round(100 * np.mean([bool(re.search(r"\w[.,!?;:](\s|$)", x)) for x in p]), 1)
            r[f"{name}|{shots_name}"] = s
            pd.DataFrame({"src": srcs, "pred": p, "pred_retok": pr}).to_csv(OUT / f"pred_{name}_qwen_{shots_name}.csv", index=False)
            print(name, shots_name, s)
    RESULTS["qwen"] = r; save_results(); del m; gc.collect(); torch.cuda.empty_cache()


In [ ]:
# ---------------------------------------------------------------- P7: JFLEG with the OFFICIAL GLEU (the paper used NLTK's MT GLEU)
if RUN["jfleg"] and "jfleg" not in RESULTS and budget_ok(0.4, "JFLEG"):
    r = {"do_nothing": jfleg_gleu([RETOK["jfleg"](x) for x in JF_SRC[:JF_N]], JF_N)}
    if HAVE_CKPT:
        tok, m = load_t5(S2_DIR); js = JF_SRC[:JF_N]
        p = decode_variants(tok, m, js, (0.7,), (True,), (0.0,), desc="jfleg")[(0.7, True, 0.0)]
        pr = [RETOK["jfleg"](x) for x in p]; r["ours_final"] = jfleg_gleu(pr, JF_N)
        pd.DataFrame({"src": js, "pred": p, "pred_retok": pr}).to_csv(OUT / "pred_jfleg_final.csv", index=False)
        del m; gc.collect(); torch.cuda.empty_cache()
    RESULTS["jfleg"] = r; save_results(); print(r)


In [ ]:
# ---------------------------------------------------------------- P8: latency + memory, same GPU for every system (R2.5)
# One T4 (cuda:0). Batch-1 latency (median / p90 ms per sentence) and batch-32 throughput on the same CoNLL sentences.
def measure(name, fn_batch, params, n=LAT_N, warm=LAT_WARM):
    srcs = CONLL_SRC[:n]; torch.cuda.synchronize(); fn_batch(srcs[:warm])
    torch.cuda.reset_peak_memory_stats(0); lat = []
    for s in srcs:
        torch.cuda.synchronize(); t = time.perf_counter(); fn_batch([s]); torch.cuda.synchronize(); lat.append(1000 * (time.perf_counter() - t))
    torch.cuda.synchronize(); t = time.perf_counter()
    for i in range(0, n, 32): fn_batch(srcs[i:i + 32])
    torch.cuda.synchronize(); thr = n / (time.perf_counter() - t)
    return {"system": name, "params_M": round(params / 1e6, 1), "lat_median_ms": round(float(np.median(lat)), 1),
            "lat_p90_ms": round(float(np.percentile(lat, 90)), 1), "throughput_sent_s_b32": round(thr, 2),
            "peak_mem_MB": round(torch.cuda.max_memory_allocated(0) / 2**20, 0)}
def nparams(m): return sum(p.numel() for p in m.parameters())
if RUN["latency"] and "latency" not in RESULTS and torch.cuda.is_available() and budget_ok(0.8, "latency"):
    rows = []
    if HAVE_CKPT:
        tok, m = load_t5(S2_DIR)
        rows.append(measure("ours, final pipeline (beam 5 + refinement + threshold)",
                            lambda b: decode_variants(tok, m, b, (0.7,), (True,), (0.0,)), nparams(m)))
        rows.append(measure("ours, greedy single pass", lambda b: t5_gen(tok, m, b, {"num_beams": 1, "length_penalty": 1.0}), nparams(m)))
        del m; gc.collect(); torch.cuda.empty_cache()
    for mid in GEC_BASELINES:
        m = None
        try:
            if "gector" in mid:
                from gector import predict, load_verb_dict
                tok = AutoTokenizer.from_pretrained(mid); m = load_gector(mid, "cuda:0"); e, d = load_verb_dict(str(VERB))
                f = lambda b: predict(m, tok, b, e, d, keep_confidence=0.3, min_error_prob=0.6, n_iteration=5, batch_size=32)
            else:
                tok = AutoTokenizer.from_pretrained(mid); m = AutoModelForSeq2SeqLM.from_pretrained(mid).to("cuda:0").eval()
                def f(b):
                    with torch.no_grad():
                        x = tok(b, return_tensors="pt", padding=True, truncation=True, max_length=128).to("cuda:0")
                        return m.generate(**x, num_beams=5, max_length=128)
            rows.append(measure(mid, f, nparams(m)))
        except Exception as ex: rows.append({"system": mid, "error": repr(ex)})
        m = None; gc.collect(); torch.cuda.empty_cache()
    try:
        tok = AutoTokenizer.from_pretrained(LLM_NAME); tok.padding_side = "left"
        if tok.pad_token is None: tok.pad_token = tok.eos_token
        m = AutoModelForCausalLM.from_pretrained(LLM_NAME, torch_dtype=torch.float16, device_map={"": 0}).eval()
        shots = RESULTS.get("qwen", {}).get("exemplars", [])
        rows.append(measure(f"{LLM_NAME}, 4-shot greedy", lambda b: llm_gen(m, tok, b, shots), nparams(m)))
        del m; gc.collect(); torch.cuda.empty_cache()
    except Exception as ex: rows.append({"system": LLM_NAME, "error": repr(ex)})
    RESULTS["latency"] = {"gpu": torch.cuda.get_device_name(0), "rows": rows}; save_results()
    print(pd.DataFrame(rows).to_string())


In [ ]:
# ---------------------------------------------------------------- P9: is reference-distance "harm" genuinely harmful? (R2.6)
# Blind pairwise judgement by an instruction model that never sees the reference: source vs our output, both orders.
# Sample = the paper's stored BEA-dev outputs of the final system, stratified by the five-way class.
pip("bitsandbytes")
from transformers import BitsAndBytesConfig
J_SYS = "You are an expert English editor. You compare two versions of the same learner sentence."
def j_better(a, b): return (f"Version 1: {a}\nVersion 2: {b}\n\nWhich version is better English (more grammatical and "
                            "natural) while keeping the same meaning? Reply with exactly one word: 1, 2, or same.")
def j_meaning(a, b): return (f"Original: {a}\nEdited: {b}\n\nDoes the edited sentence keep the meaning of the original? "
                             "Reply with exactly one word: yes or no.")
@torch.no_grad()
def judge(m, tok, users):
    out = []
    for i in range(0, len(users), 16):
        P = [tok.apply_chat_template([{"role": "system", "content": J_SYS}, {"role": "user", "content": u}],
                                     tokenize=False, add_generation_prompt=True) for u in users[i:i + 16]]
        enc = tok(P, return_tensors="pt", padding=True).to(m.device)
        g = m.generate(**enc, max_new_tokens=3, do_sample=False, pad_token_id=tok.pad_token_id)
        out += [tok.decode(x[enc.input_ids.shape[1]:], skip_special_tokens=True).strip().lower() for x in g]
    return out
if RUN["judge"] and "judge" not in RESULTS and budget_ok(0.8, "judge"):
    src, pred = STORED["bea_dev_src"], STORED["bea_dev_pred"]; tgt = BEA_TGT[:len(src)]
    assert src == BEA_SRC[:len(src)], "stored predictions are not in BEA-dev gold order"
    cls = []
    for s, t, p in zip(src, tgt, pred):
        if p.strip() == t.strip(): cls.append("perfect")
        elif p.strip() == s.strip(): cls.append("unchanged")
        else:
            ds, dp = lev(s, t), lev(p, t); cls.append("improved" if dp < ds else ("lateral" if dp == ds else "harm"))
    rng = random.Random(7); items = []
    for c, n in JUDGE_N.items():
        pool = [i for i, x in enumerate(cls) if x == c]; items += [(i, c) for i in rng.sample(pool, min(n, len(pool)))]
    try:
        q = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
        tok = AutoTokenizer.from_pretrained(JUDGE_NAME); m = AutoModelForCausalLM.from_pretrained(JUDGE_NAME, quantization_config=q, device_map={"": 0}); used = JUDGE_NAME + " (4-bit nf4)"
    except Exception as ex:
        print("7B judge unavailable:", ex); tok = AutoTokenizer.from_pretrained(JUDGE_FALLBACK)
        m = AutoModelForCausalLM.from_pretrained(JUDGE_FALLBACK, torch_dtype=torch.float16, device_map={"": 0}); used = JUDGE_FALLBACK + " (fp16)"
    tok.padding_side = "left"
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    a1 = judge(m, tok, [j_better(src[i], pred[i]) for i, _ in items])     # output shown as version 2
    a2 = judge(m, tok, [j_better(pred[i], src[i]) for i, _ in items])     # output shown as version 1
    mean = judge(m, tok, [j_meaning(src[i], pred[i]) for i, _ in items])
    def norm(x): return "1" if x.startswith("1") else "2" if x.startswith("2") else "same" if x.startswith("same") else "?"
    rows = []
    for (i, c), x, y, z in zip(items, a1, a2, mean):
        x, y = norm(x), norm(y)
        v = ("better" if (x == "2" and y == "1") else "worse" if (x == "1" and y == "2") else
             "equal" if (x == "same" and y == "same") else "inconsistent")
        rows.append({"bea_dev_index": i, "class": c, "src": src[i], "pred": pred[i], "ref": tgt[i],
                     "order_pred_second": x, "order_pred_first": y, "verdict_for_output": v,
                     "meaning_kept": z.startswith("yes")})
    df = pd.DataFrame(rows); df.to_csv(OUT / "judge_items.csv", index=False)
    summ = {c: {**df[df["class"] == c]["verdict_for_output"].value_counts(normalize=True).round(3).to_dict(),
                "meaning_kept": round(float(df[df["class"] == c]["meaning_kept"].mean()), 3), "n": int((df["class"] == c).sum())}
            for c in JUDGE_N}
    RESULTS["judge"] = {"judge_model": used, "summary": summ, "class_counts_all_dev": dict(Counter(cls))}
    save_results(); print(json.dumps(RESULTS["judge"], indent=1))
    # blinded sheet for an OPTIONAL human check by the authors (key kept separately)
    hs = df.groupby("class", group_keys=False).apply(lambda d: d.sample(min(50, len(d)), random_state=1)).reset_index(drop=True)
    flip = [random.Random(100 + k).random() < 0.5 for k in range(len(hs))]
    pd.DataFrame({"item": range(len(hs)), "sentence_1": [p if f else s for s, p, f in zip(hs.src, hs.pred, flip)],
                  "sentence_2": [s if f else p for s, p, f in zip(hs.src, hs.pred, flip)],
                  "better (1/2/same)": "", "meaning_same (yes/no)": ""}).to_csv(OUT / "human_check_sheet.csv", index=False)
    pd.DataFrame({"item": range(len(hs)), "output_is": ["sentence_1" if f else "sentence_2" for f in flip],
                  "class": hs["class"], "bea_dev_index": hs.bea_dev_index}).to_csv(OUT / "human_check_KEY.csv", index=False)
    del m; gc.collect(); torch.cuda.empty_cache()


In [ ]:
# ---------------------------------------------------------------- export everything except model weights
zp = OUT / EXPORT_NAME
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.iterdir()):
        if p.is_file() and not p.name.startswith("_") and p.suffix in (".json", ".csv", ".txt", ".zip", ".png", ".tsv", ".gz") \
                and p.name != EXPORT_NAME:
            z.write(p, p.name)
print(f"wrote {zp} ({zp.stat().st_size/1e6:.1f} MB) -- download this file and send it back")
print(json.dumps({k: v for k, v in RESULTS.items() if not isinstance(v, (list,))}, indent=1, default=str)[:4000])
